# Figure 4 — Q3: how far does confounding carry a correction before it breaks?

Batch-phenotype confounding is a designed factor here, swept from 0 to 1. Every panel asks the same question at a different angle: where does each method stop working?

**Each panel is defined and drawn in its own cell.** Edit the function, re-run that one cell, see the result. The last cell assembles all of them at the submission size (180 x 215 mm) and writes the PDF that goes to the journal.

Run the Setup, Shared helpers and Data cells first; after that the panel cells are independent of each other.

## Setup

In [ ]:
# Everything the panels read is relative to benchmark/, so make that the working directory
# whether the notebook was opened from benchmark/ or from benchmark/figures/.
if (basename(getwd()) == "figures") setwd("..")
stopifnot("run this from benchmark/ or benchmark/figures/" = file.exists("config/methods.tsv"))

# _panels.R holds what every figure shares: the method inventory and its ordering, the metric
# catalogue, the two normalisations used for ranking, the theme, and save_figure().
source("figures/_panels.R")

results_dir <- "results"

## Shared helpers

Small pieces more than one panel below needs.

In [ ]:
batch_metrics <- function() c("R2_batch_ait", "R2_batch_bc", "R2_batch_jac", "ASW_batch", "iLISI")

bio_metrics <- function() c("R2_pheno_ait", "R2_pheno_bc", "R2_pheno_jac", "ASW_pheno", "cLISI")

# Per dataset and method, the scaled score of each metric family, with the scenario parameters kept.
scored_by_scenario <- function(scored) {
  parameters <- scored[!duplicated(scored$dataset), c("dataset", "scenario", "family", "cramers_v",
                                                      "conf", "template", "da_prop", "affected")]
  families <- bind_rows(
    family_score(scored, batch_metrics(), by = c("dataset", "method")),
    family_score(scored, bio_metrics(), by = c("dataset", "method")))
  left_join(families, parameters, by = "dataset", suffix = c("", "_scenario"))
}

# Eighteen curves in one palette are eighteen shades nobody can tell apart, so only a few are drawn
# in colour: the two baselines, which bound the panel, and the methods whose score actually moves
# across the gradient. The rest stay as grey context -- still visible as a band, no longer claiming
# to be identifiable.
gradient_highlight <- function(curves, n = 4) {
  swing <- curves %>%
    group_by(method, family, conf) %>%
    summarise(score = mean(score, na.rm = TRUE), .groups = "drop") %>%
    group_by(method, family) %>%
    summarise(swing = abs(score[which.max(conf)] - score[which.min(conf)]), .groups = "drop") %>%
    group_by(method) %>%
    summarise(swing = max(swing), .groups = "drop")
  c("raw", "oracle", setdiff(swing$method[order(-swing$swing)], c("raw", "oracle"))[seq_len(n)])
}

gradient_colours <- function(highlight, methods_table = method_table())
  setNames(c("grey35", "black", RColorBrewer::brewer.pal(max(3, length(highlight) - 2), "Dark2")),
           methods_table$label[match(highlight, methods_table$method)])

# Marks each curve with its own label where it is highlighted and with one shared level otherwise,
# so a single colour scale carries both the named methods and the grey background.
by_highlight <- function(curves, highlight, named)
  transform(curves, shown = factor(ifelse(curves$method %in% highlight, as.character(curves$label),
                                          "other methods"), c(named, "other methods")))

## Data

In [ ]:
scored <- read_scores(file.path(results_dir, "scores.tsv"))
simulated <- scored[scored$regime == "sim" & scored$status == "ok", ]
by_scenario <- scored_by_scenario(simulated)
real_by_dataset <- bind_rows(
  family_score(scored[scored$regime == "real" & scored$status == "ok", ], batch_metrics(),
               by = c("dataset", "method")),
  family_score(scored[scored$regime == "real" & scored$status == "ok", ], bio_metrics(),
               by = c("dataset", "method")))
confounding <- dataset_confounding(results_dir, unique(scored$dataset[scored$regime == "real"]))
names(confounding) <- sub("^real/", "", unique(scored$dataset[scored$regime == "real"]))

# Eighteen curves in one palette are eighteen shades nobody can tell apart: the two baselines and
# the four methods that actually move get colour, the rest stay grey.
sweep_scores <- by_scenario[by_scenario$family_scenario == "sweep", ]
highlight <- gradient_highlight(sweep_scores)
colours <- gradient_colours(highlight)

cat("highlighted:", paste(highlight, collapse = ", "), "\n")

## a. Performance along the gradient

Scaled score against designed confounding, split by assay and by metric family. Grey lines are the other twelve methods.

In [ ]:
# The gradient is indexed by the designed confounding level, not by each replicate's realised
# Cramer's V: with 20 replicates per level the realised values scatter around the design point and
# would turn five points into a hundred.
panel_gradient <- function(sweep_scores, highlight, colours, methods_table = method_table()) {
  curves <- sweep_scores %>%
    group_by(method, family, template, conf) %>%
    summarise(spread = sd(score, na.rm = TRUE), score = mean(score, na.rm = TRUE), .groups = "drop")
  curves <- by_highlight(label_methods(curves, methods_table), highlight, names(colours))
  ggplot(curves, aes(conf, score, colour = shown, group = label)) +
    geom_line(data = ~ subset(.x, shown == "other methods"), linewidth = 0.25, alpha = 0.7) +
    geom_errorbar(data = ~ subset(.x, shown != "other methods"),
                  aes(ymin = score - spread, ymax = score + spread), width = 0, linewidth = 0.25,
                  alpha = 0.5) +
    geom_line(data = ~ subset(.x, shown != "other methods"), linewidth = 0.5) +
    geom_point(data = ~ subset(.x, shown != "other methods"), size = 0.8) +
    facet_grid(family ~ template, labeller = as_labeller(
      c(batch = "batch removal", bio = "signal retention", `16s.tsv` = "16S", `mgx.tsv` = "metagenome",
        `16s` = "16S", mgx = "metagenome"))) +
    scale_colour_manual(values = c(colours, `other methods` = "grey80"), name = NULL) +
    scale_x_continuous(breaks = c(0, 0.3, 0.6, 0.9, 1), labels = c("0", "0.3", "0.6", "0.9", "1")) +
    scale_y_continuous(breaks = c(0, 0.5, 1)) +
    guides(colour = guide_legend(nrow = 1, override.aes = list(linewidth = 0.8, size = 0.9))) +
    labs(x = "designed batch-phenotype confounding", y = "scaled score (mean +- sd over replicates)",
         title = "Performance along the confounding gradient") +
    theme_bench() +
    theme(legend.position = "bottom", legend.key.width = unit(5, "mm"),
          panel.grid.major.x = element_blank(),
          panel.border = element_rect(fill = NA, colour = "grey75", linewidth = 0.25),
          axis.line = element_blank())
}

show_panel(panel_gradient(sweep_scores, highlight, colours), 180, 80)

## b. Overcorrection index

Biological signal lost per unit of batch signal removed. The axis is symmetric because the ratio goes negative, and that is the finding: apparent phenotype signal *grew* while the batch term shrank. The four highlighted methods are the ones coloured in panel a, named here at the end of their lines.

In [ ]:
# How much biological signal a method gives up for each unit of batch signal it removes, as a median
# over replicates (the ratio is unstable wherever a method barely moved the batch term).
#
# The axis is symmetric around zero because the ratio goes negative, and that is the finding rather
# than an artefact: a negative index means the apparent phenotype signal *grew* while the batch term
# shrank, which is batch structure being moved onto the phenotype axis rather than removed. A log
# axis would silently drop every one of those points.
panel_overcorrection <- function(simulated, highlight, colours, methods_table = method_table()) {
  reference <- simulated[simulated$method == "raw", c("dataset", "R2_batch_ait", "R2_pheno_ait")]
  names(reference)[2:3] <- c("raw_batch", "raw_pheno")
  # oracle has no batch term to remove, so the ratio is not defined for it
  corrected <- inner_join(simulated[!simulated$method %in% c("raw", "oracle"), ], reference, by = "dataset")
  corrected$index <- (corrected$raw_pheno - corrected$R2_pheno_ait) /
                     (corrected$raw_batch - corrected$R2_batch_ait)
  summarised <- corrected %>%
    group_by(method, conf) %>%
    summarise(index = median(index, na.rm = TRUE), .groups = "drop") %>%
    filter(!is.na(index))   # a method with no result at a level has no point there
  marked <- by_highlight(label_methods(summarised, methods_table), highlight, names(colours))
  line_ends <- marked %>% filter(shown != "other methods") %>% group_by(label) %>%
    filter(conf == max(conf)) %>% ungroup()
  ggplot(marked, aes(conf, index, colour = shown, group = label)) +
    geom_hline(yintercept = c(0, 1), linetype = c("solid", "22"), linewidth = 0.25,
               colour = c("grey50", "grey50")) +
    geom_line(data = ~ subset(.x, shown == "other methods"), linewidth = 0.25, alpha = 0.7) +
    geom_line(data = ~ subset(.x, shown != "other methods"), linewidth = 0.5) +
    geom_point(data = ~ subset(.x, shown != "other methods"), size = 0.8) +
    ggrepel::geom_text_repel(data = line_ends, aes(label = label), size = 1.8, colour = "grey15",
                             direction = "y", hjust = 0, nudge_x = 0.06, xlim = c(1.04, NA),
                             segment.size = 0.2, segment.colour = "grey60", min.segment.length = 0,
                             box.padding = 0.1, seed = 1) +
    scale_colour_manual(values = c(colours, `other methods` = "grey80"), guide = "none") +
    scale_x_continuous(breaks = c(0, 0.3, 0.6, 0.9, 1), labels = c("0", "0.3", "0.6", "0.9", "1"),
                       expand = expansion(mult = c(0.04, 0.32))) +
    scale_y_continuous(trans = scales::pseudo_log_trans(sigma = 0.01),
                       breaks = c(-10, -1, -0.1, 0, 0.1, 1, 10), labels = scales::label_number()) +
    labs(x = "designed confounding", y = "signal lost / batch removed",
         title = "Overcorrection index",
         subtitle = "above 0: signal lost with the batch; below 0: batch moved onto the phenotype") +
    theme_bench() + theme(panel.grid.major.x = element_blank())
}

show_panel(panel_overcorrection(simulated[simulated$family == "sweep", ], highlight, colours), 100, 68)

## c. Breaking point

Where each method first keeps less than half the signal the oracle still shows. Most never do, and sit in the shaded column.

In [ ]:
# The point on the gradient where a method keeps less than half of the signal the oracle still shows.
panel_breakpoint <- function(sweep_simulated, methods_table = method_table()) {
  oracle <- sweep_simulated[sweep_simulated$method == "oracle", c("dataset", "R2_pheno_ait")]
  names(oracle)[2] <- "oracle_pheno"
  retained <- inner_join(sweep_simulated[sweep_simulated$method != "oracle", ], oracle, by = "dataset")
  retained$retention <- retained$R2_pheno_ait / retained$oracle_pheno
  breaks <- retained %>%
    group_by(method, conf) %>%
    summarise(retention = mean(retention, na.rm = TRUE), .groups = "drop") %>%
    group_by(method) %>%
    summarise(breakpoint = suppressWarnings(min(conf[retention < 0.5])), .groups = "drop")
  never <- 1.2                                              # where "never broke" is drawn
  breaks$breakpoint[!is.finite(breaks$breakpoint)] <- never
  ggplot(label_methods(breaks, methods_table), aes(breakpoint, label)) +
    annotate("rect", xmin = never - 0.1, xmax = never + 0.1, ymin = -Inf, ymax = Inf, fill = "grey94") +
    geom_segment(aes(x = 0, xend = breakpoint, yend = label), colour = "grey80", linewidth = 0.3) +
    geom_point(aes(colour = track), size = 1.6) +
    scale_colour_manual(values = track_colours(), guide = "none") +
    scale_x_continuous(breaks = c(0, 0.3, 0.6, 0.9, never), labels = c("0", "0.3", "0.6", "0.9", "never"),
                       limits = c(0, never + 0.1), expand = expansion(0)) +
    scale_y_discrete(limits = rev) +
    labs(x = "designed confounding", y = NULL, title = "Breaking point",
         subtitle = "where less than half of the oracle's signal survives") +
    theme_bench() + theme(panel.grid.major = element_blank())
}

show_panel(panel_breakpoint(simulated[simulated$family == "sweep", ]), 80, 68)

## d. Invented structure in the null scenarios

With no biological signal planted at all, how much phenotype variance does each method manufacture? One dot per designed confounding level.

In [ ]:
panel_invented_signal <- function(null_scores, methods_table = method_table()) {
  invented <- null_scores %>% group_by(method, conf) %>%
    summarise(r2 = mean(R2_pheno_ait, na.rm = TRUE), .groups = "drop") %>%
    filter(!is.na(r2))
  invented$confounding <- factor(invented$conf)
  ggplot(label_methods(invented, methods_table), aes(r2, label)) +
    geom_line(aes(group = label), colour = "grey75", linewidth = 0.35) +
    geom_point(aes(fill = confounding), shape = 21, colour = "white", stroke = 0.25, size = 1.7) +
    scale_fill_manual(values = level_colours(), name = "designed confounding") +
    scale_x_continuous(expand = expansion(mult = c(0.03, 0.05)), limits = c(0, NA)) +
    scale_y_discrete(limits = rev) +
    labs(x = "R2 of a phenotype that carries no signal", y = NULL,
         title = "Invented structure in the null scenarios",
         subtitle = "a missing dot: the method returned no result at that level") +
    theme_bench() + theme(legend.position = "bottom")
}

show_panel(panel_invented_signal(simulated[simulated$family == "null" & simulated$method != "oracle", ]), 80, 75)

## e. The same measures on real data

Balanced against confounded real designs, split at Cramer's V = 0.2. Each method is one pair of dots: batch removal and signal retention.

In [ ]:
panel_real_anchors <- function(real_scores, confounding, methods_table = method_table()) {
  anchored <- real_scores
  anchored$v <- confounding[sub("^real/", "", anchored$dataset)]
  anchored$end <- ifelse(anchored$v < 0.2, "balanced designs (V < 0.2)", "confounded designs (V >= 0.2)")
  summarised <- anchored %>%
    group_by(method, family, end) %>%
    summarise(score = mean(score, na.rm = TRUE), .groups = "drop")
  ggplot(label_methods(summarised, methods_table), aes(score, label)) +
    geom_line(aes(group = label), colour = "grey75", linewidth = 0.35) +
    geom_point(aes(fill = family), shape = 21, colour = "white", stroke = 0.25, size = 1.7) +
    facet_wrap(~ end, nrow = 1) +
    scale_fill_manual(values = setNames(pair_colours(), c("batch", "bio")), name = NULL,
                      labels = c(batch = "batch removal", bio = "signal retention")) +
    scale_x_continuous(limits = c(0, 1), breaks = c(0, 0.5, 1), labels = c("0", "0.5", "1")) +
    scale_y_discrete(limits = rev) +
    labs(x = "scaled score", y = NULL, title = "The same measures on real data") +
    theme_bench() + theme(legend.position = "bottom")
}

show_panel(panel_real_anchors(real_by_dataset, confounding), 100, 75)

## Assembled figure

Everything above, in the layout that goes to the journal: 180 x 215 mm, one page.
The panels shrink here — that is the real submission size, and the font sizes above were
chosen for it.

In [ ]:
sweep_row <- (panel_overcorrection(simulated[simulated$family == "sweep", ], highlight, colours) |
              panel_breakpoint(simulated[simulated$family == "sweep", ])) +
  plot_layout(widths = c(1.3, 1))
anchor_row <- (panel_invented_signal(simulated[simulated$family == "null" & simulated$method != "oracle", ]) |
               panel_real_anchors(real_by_dataset, confounding)) +
  plot_layout(widths = c(1, 1.5))

figure <- assemble(panel_gradient(sweep_scores, highlight, colours) / sweep_row / anchor_row +
                   plot_layout(heights = c(1.25, 1, 1.1)))

save_figure(figure, "results/figures/fig4.pdf", 180, 215)
show_panel(figure, 180, 215, scale = 1.5)